In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# ==========================================
# 1. 데이터 로드 (파일 경로는 환경에 맞게 수정하세요)
# ==========================================
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# ==========================================
# 2. 전처리 및 피처 엔지니어링 함수
# ==========================================
def preprocess_pipeline(df, is_train=True):
    df = df.copy()
    
    # [데이터 정제] TotalCharges 숫자 변환 및 결측치 처리
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(df['MonthlyCharges'])
    
    # [타겟 변수 수치화] Train 데이터인 경우에만 적용
    if is_train and 'Churn' in df.columns:
        df['Churn_num'] = df['Churn'].map({'Yes': 1, 'No': 0})
    
    # [FE 1] 가족 결합도 (IsFamily): 배우자와 부양가족이 모두 있는 경우
    df['IsFamily'] = ((df['Partner'] == 'Yes') & (df['Dependents'] == 'Yes')).astype(int)
    
    # [FE 2] 요금 급증 확인 (ChargeDiff): 현재 요금 - 과거 평균 요금
    df['AvgMonthlyCharge'] = df['TotalCharges'] / (df['tenure'] + 1)
    df['ChargeDiff'] = df['MonthlyCharges'] - df['AvgMonthlyCharge']
    
    # [FE 3] 서비스 가입 개수 (ServiceCount): 부가 서비스 이용 합계
    service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
    df['ServiceCount'] = (df[service_cols] == 'Yes').sum(axis=1)
    
    return df

# 전처리 적용
train = preprocess_pipeline(train, is_train=True)
test = preprocess_pipeline(test, is_train=False)


In [ ]:
# ==========================================
# 3. 학습 준비 (피처 선택 및 인코딩)
# ==========================================
selected_features = [
    'tenure', 'Contract', 'OnlineSecurity', 'TechSupport', 
    'InternetService', 'PaymentMethod', 'MonthlyCharges', 
    'IsFamily', 'ChargeDiff', 'ServiceCount'
]

# 원-핫 인코딩 적용
X = pd.get_dummies(train[selected_features], drop_first=True)
y = train['Churn_num']
X_test = pd.get_dummies(test[selected_features], drop_first=True)

# 학습 데이터와 테스트 데이터의 컬럼 일치시키기 (reindex)
X_test = X_test.reindex(columns=X.columns, fill_value=0)

# ==========================================
# 4. 5-Fold 교차 검증 및 모델 학습
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))      # 검증 데이터 예측값 저장용
test_preds = np.zeros(len(X_test)) # 테스트 데이터 예측값(앙상블) 저장용
auc_scores = []

print("🚀 5-Fold Cross Validation & Training Started...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
    y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
    
    # 최신 XGBoost 하이퍼파라미터 설정
    model = XGBClassifier(
        n_estimators=1000, 
        learning_rate=0.02, 
        max_depth=6, 
        subsample=0.8, 
        colsample_bytree=0.8, 
        random_state=42,
        early_stopping_rounds=50, # 조기 종료 설정
        eval_metric='auc'
    )
    
    # 모델 학습
    model.fit(
        X_train_fold, y_train_fold,
        eval_set=[(X_val_fold, y_val_fold)],
        verbose=False
    )
    
    # 검증 및 테스트 데이터 예측
    fold_probs = model.predict_proba(X_val_fold)[:, 1]
    oof_preds[val_idx] = fold_probs
    
    # 테스트 데이터 예측값 누적 (나중에 5로 나누어 평균값 산출)
    test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    # 각 폴드의 점수 기록
    fold_auc = roc_auc_score(y_val_fold, fold_probs)
    auc_scores.append(fold_auc)

# 최종 결과 리포트
print("\n" + "="*30)
print(f"✅ Final Mean ROC-AUC: {np.mean(auc_scores):.4f}")
print("="*30)

🚀 5-Fold Cross Validation & Training Started...
Fold 1 ROC-AUC: 0.9126
Fold 2 ROC-AUC: 0.9137
Fold 3 ROC-AUC: 0.9133
Fold 4 ROC-AUC: 0.9144
Fold 5 ROC-AUC: 0.9117

✅ Final Mean ROC-AUC: 0.9131


In [23]:
# ==========================================
# 5. 최종 제출 파일 생성
# ==========================================
submission = pd.DataFrame({
    'id': test['id'],
    'Churn': test_preds # 5개 모델의 평균 확률값
})

submission.to_csv('final_ensemble_submission.csv', index=False)
print("\n📦 'final_ensemble_submission.csv'가 성공적으로 저장되었습니다!")


📦 'final_ensemble_submission.csv'가 성공적으로 저장되었습니다!
